# 07 — Noise Robustness and Cross-Function Generalization

Run independently. These are **separate secondary analyses**, outside the native
ranking. This notebook does not launch any evaluations or synthesis. It loads
results matching current DB-selected champion code. Missing/incomplete transfer
cells stay grey. Native diagonal and baseline references are reused.

Run either export section independently: transfer matrices are a smaller workload
than the per-algorithm noise-overlay sweep. They use separate cache manifests;
changes to transfer results never invalidate native profiles or noise overlays.
Outputs are PNGs and versioned CSVs, with no inline images or HTML figures.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bootstrap").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from bootstrap.analysis import build_analysis_inputs
from notebooks.analysis.plotting.cache import build_figure_cache

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None
FORCE_EXPORT = False


In [2]:
inputs = build_analysis_inputs(include_transfer=True, dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
noise_cache = build_figure_cache(inputs, "noise", force=FORCE_EXPORT)
transfer_cache = build_figure_cache(inputs, "transfer", force=FORCE_EXPORT)
print(f"Loaded {len(inputs.noise_records)} frozen-noise references and {len(inputs.transfer_records)} transfer/native-reference conditions.")


Loaded 480 frozen-noise references and 60 transfer/native-reference conditions.


## Cross-function transfer matrix and aggregate intervals

Score the returned point with the target's clean objective and optimum. Fixed-target
off-diagonal aggregates weight complete source–target conditions equally. Intervals
bootstrap whole conditions; unavailable conditions are not converted to failures.


In [3]:
from notebooks.analysis.plotting.generalization import export_cross_function
transfer_reliability = export_cross_function(inputs, transfer_cache)
display(transfer_reliability)


Cross-function tables and PNG matrices exported; native rankings are unchanged.


,Model,Strategy,Solver,Source Problem,Target Problem,Dim,Noise Std,Code Hash,Trials,Complete,Successes,Success Rate,Secondary Success Rate,CI Lower,CI Upper,Median Terminal Error
0,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,baseline,deepseek_coder_v2_lite_instruct_q4_k_m_baseline,1,1,10,0.0,6a16baf0d79ec5c1650a2963d22837fd480d7df7a3ed50...,20,True,0,0.00,0.00,0.000000,0.161125,5.989779e+01
1,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,qwen_14b_baseline,1,1,10,0.0,9e42a55b5941235dcd23aca13ef16fa1a920ce4d568855...,20,True,20,1.00,1.00,0.838875,1.000000,0.000000e+00
2,qwen2.5-coder-32b-instruct-q4_k_m.gguf,baseline,qwen_32b_baseline,1,1,10,0.0,63dfb1984573c7ae38998b9d2b54d16b4d8270051b6854...,20,True,0,0.00,0.00,0.000000,0.161125,3.447636e+01
3,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,baseline,deepseek_coder_v2_lite_instruct_q4_k_m_baseline,11,11,10,0.0,38449b11440495bd92db924b06c036bb56274eb4fc8ba3...,20,True,0,0.00,0.00,0.000000,0.161125,6.027797e+01
4,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,qwen_14b_baseline,11,11,10,0.0,7026882b44408a2cf10fcaa6964ddb57c5c93bb480109d...,20,True,0,0.00,0.00,0.000000,0.161125,2.462213e-01
5,qwen2.5-coder-32b-instruct-q4_k_m.gguf,baseline,qwen_32b_baseline,11,11,10,0.0,a165c208824602e45356f7b0404d373dc6027ddd37d4bd...,20,True,0,0.00,0.95,0.000000,0.161125,2.666027e-03
6,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,baseline,deepseek_coder_v2_lite_instruct_q4_k_m_baseline,15,15,10,0.0,0b65744a0faafafe56d7c3ae94221b7e82ee28569973d8...,20,True,0,0.00,0.00,0.000000,0.161125,3.490878e+01
7,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,qwen_14b_baseline,15,15,10,0.0,a34122f85c34e536602dc25f7b3da3e4e997a0d47fd3b2...,20,True,0,0.00,0.00,0.000000,0.161125,7.502772e+00
8,qwen2.5-coder-32b-instruct-q4_k_m.gguf,baseline,qwen_32b_baseline,15,15,10,0.0,978e3375079eb3e9285190f0106a61f40591a9ec7babc8...,20,True,0,0.00,0.00,0.000000,0.161125,2.971325e+01
9,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,baseline,deepseek_coder_v2_lite_instruct_q4_k_m_baseline,21,21,10,0.0,6562ea26a652d1105e772a35b33a2c568daec931e4264b...,20,True,0,0.00,0.00,0.000000,0.161125,8.035440e+01


## Frozen-champion noise overlays (optional slower sweep)

Only identical clean champion code is compared across noise levels. Both curves use
the filtered frozen-champion dataset. ECDFs share the same primary/secondary targets;
LLM lines are dashed and classical baseline lines are solid.


In [4]:
from notebooks.analysis.plotting.generalization import export_noise_robustness
export_noise_robustness(inputs, noise_cache)
